# Notes

code for producing GOIT pipelines summary stats, and for calculating landing page stats

this is saved as an Excel file, which Baird copies/pastes into the existing summary tables information on the drive here:
https://docs.google.com/spreadsheets/d/1OYH6D7c-D0FsL5GzBGijtkmvQCTkBUclj-UVoOieUFo/edit

In [131]:
import pandas
import numpy
import pygsheets
import datetime
import re
import pytz

In [132]:
# define the excel file to save tables in
current_time = datetime.datetime.now(pytz.timezone('US/Eastern')).strftime("%Y-%m-%d_T%H%M%S")

In [133]:
fuel_type = 'Gas'
#fuel_type = 'Oil'
#fuel_type = 'NGL'

## import data

In [134]:
gc = pygsheets.authorize(service_account_env_var='GDRIVE_API_CREDENTIALS')
#spreadsheet = gc.open_by_key('1WaBMIdfRWqSqXUw7_cKXo3RipyhPdnNN8flqEYfMZIA') # file to use for gas pipelines Dec 2023
# spreadsheet = gc.open_by_key('1foPLE6K-uqFlaYgLPAUxzeXfDO5wOOqE7tibNHeqTek') # CURRENT sheet
spreadsheet = gc.open_by_key('1OXybaZOn0f2ONB6d_J0A3SG2bJ660C2Kr8fuc5o8cjs') # dec 2024 dataset for GGIT release

gas_pipes = spreadsheet.worksheet('title', 'Gas pipelines').get_as_df(start='A3')
oil_pipes = spreadsheet.worksheet('title', 'Oil/NGL pipelines').get_as_df(start='A3')

gas_pipes = gas_pipes.drop('WKTFormat', axis=1) # delete WKTFormat column
oil_pipes = oil_pipes.drop('WKTFormat', axis=1)
pipes_df_orig = gas_pipes.copy() #pandas.concat([oil_pipes, gas_pipes], ignore_index=True)

#get other relevant sheets
country_ratios_df = spreadsheet.worksheet('title', 'Country ratios by pipeline').get_as_df()
owners_df_orig = spreadsheet.worksheet('title', 'Pipeline operators/owners').get_as_df(start='A2')

country_ratios_df = country_ratios_df.loc[country_ratios_df.Wiki!='']

# remove empty cells for pipes, owners
pipes_df_orig = pipes_df_orig.loc[pipes_df_orig['PipelineName']!='']
pipes_df_orig = pipes_df_orig.loc[pipes_df_orig['Wiki']!='']
pipes_df_orig = pipes_df_orig.loc[pipes_df_orig.Fuel==fuel_type]

owners_df_orig = owners_df_orig.loc[owners_df_orig['ProjectID']!='']
owners_df_orig = owners_df_orig.loc[owners_df_orig['Wiki']!='']
owners_df_orig = owners_df_orig.loc[owners_df_orig.Status!='N/A']

owners_df_orig.set_index('ProjectID', inplace=True)

parent_metadata_df = spreadsheet.worksheet('title', 'Parent metadata (3/3)').get_as_df(start='A3')
parent_metadata_df.set_index('Parent', inplace=True)

In [135]:
country_ratios_df.replace('--', numpy.nan, inplace=True)

owners_df_orig.replace('',numpy.nan,inplace=True)
owners_df_orig.replace('--',numpy.nan,inplace=True)

pipes_df_orig.replace('--',numpy.nan,inplace=True)

/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_12383/1702877721.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  country_ratios_df.replace('--', numpy.nan, inplace=True)
/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_12383/1702877721.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  owners_df_orig.replace('--',numpy.nan,inplace=True)
/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_12383/1702877721.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a f

In [136]:
region_df_orig = spreadsheet.worksheet('title', 'Country dictionary').get_as_df(start='A2')

#region_name = 'Global'; region_df_touse = region_df_orig.copy()
#region_name = 'AsiaGasTracker'; region_df_touse = region_df_orig.loc[region_df_orig.AsiaGasTracker=='Yes']
#region_name = 'EuroGasTracker'; region_df_touse = region_df_orig.loc[region_df_orig.EuroGasTracker=='Yes']
#region_name = 'AfricaGasTracker'; region_df_touse = region_df_orig.loc[region_df_orig.AfricaGasTracker=='Yes']
region_name = 'LatinAmericaTracker'; region_df_touse = region_df_orig.loc[region_df_orig.LatinAmericaTracker=='Yes']
#region_df_agt.copy()

#region_df_touse = region_df_orig.copy()

In [137]:
region_df_touse_cleaned = region_df_touse.loc[(region_df_touse.Region!='--')&
                                            (region_df_touse.SubRegion!='--')]
multiindex_region_subregion = region_df_touse_cleaned.groupby(['Region','SubRegion'])['Country'].count().index
multiindex_region_subregion

MultiIndex([('Americas', 'Latin America and the Caribbean')],
           names=['Region', 'SubRegion'])

## file names with regional specifics

In [138]:
if fuel_type=='Gas':
    excel_writer = pandas.ExcelWriter(region_name+'-summary-sheets-gas-pipelines-'+str(datetime.date.today())+'.xlsx')
if fuel_type=='NGL':
    excel_writer = pandas.ExcelWriter(region_name+'-summary-sheets-NGL-pipelines-'+str(datetime.date.today())+'.xlsx')
if fuel_type=='Oil':
    excel_writer = pandas.ExcelWriter(region_name+'-summary-sheets-Oil-pipelines-'+str(datetime.date.today())+'.xlsx')

### create country-specific dataframes for region, country_ratios_df, owners_df

In [139]:
country_ratios_df_touse = country_ratios_df.loc[country_ratios_df['Country'].str.contains(
                                            '|'.join(region_df_touse['Country'].tolist()))]

# owners_df_touse = owners_df_orig.loc[owners_df_orig['Countries'].str.contains(
#                                             '|'.join(region_df_touse['Country'].tolist()))]

pipes_df_touse = pipes_df_orig.loc[pipes_df_orig['Countries'].str.contains(
                                            '|'.join(region_df_touse['Country'].tolist()))]

/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_12383/3574110634.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  country_ratios_df_touse = country_ratios_df.loc[country_ratios_df['Country'].str.contains(
/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_12383/3574110634.py:7: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  pipes_df_touse = pipes_df_orig.loc[pipes_df_orig['Countries'].str.contains(


In [140]:
country_ratios_df

,PipelineName,SegmentName,ProjectID,Country,LengthEstimateKmByCountry,LengthPerCountryFraction,Region,SubRegion,RegionOld,PipelineBubbleRegion,...,Parent,H2Status,H2Type,CancelledYear,ProposalYear,ConstructionYear,ShelvedYear,StartYearEarliest,StartCountry,EndCountry
0,Alberta Clipper Oil Pipeline,,P0001,Canada,1066.328784,0.680000,Americas,Northern America,North America,North America,...,Enbridge Inc [100.00%],NaN,NaN,,,,,2010.0,Canada,United States
1,Alberta Clipper Oil Pipeline,,P0001,United States,512.042577,0.320000,Americas,Northern America,North America,North America,...,Enbridge Inc [100.00%],NaN,NaN,,,,,2010.0,Canada,United States
2,Athabasca Oil Pipeline,,P0002,Canada,522.239984,1.000000,Americas,Northern America,North America,North America,...,Enbridge Inc [88.43%]; unknown [unknown %],NaN,NaN,,,,,1999.0,Canada,Canada
3,Bakken Expansion Pipeline,,P0004,Canada,150.276903,0.595268,Americas,Northern America,North America,North America,...,Enbridge Inc [100.00%],NaN,NaN,,,,,2013.0,United States,Canada
4,Bakken Expansion Pipeline,,P0004,United States,102.1756,0.400000,Americas,Northern America,North America,North America,...,Enbridge Inc [100.00%],NaN,NaN,,,,,2013.0,United States,Canada
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6594,Nigeria–Libya Gas Pipeline,,P7102,Niger,1241.350811,0.389414,Africa,Sub-Saharan Africa,Sub-Saharan Africa,Africa,...,unknown [unknown %],NaN,,,2024,,,NaN,Nigeria,Libya
6595,Pomeranian Hydrogen Cluster,SYSTEM/NETWORK INFO,P7103,Poland,561.549529,0.999882,Europe,Eastern Europe,Europe,Europe,...,unknown [unknown %],NaN,new AND converted segments,,,,,2029.0,Poland,Poland
6596,Pomeranian Hydrogen Cluster,SYSTEM/NETWORK INFO,P7103,Germany,0.066229,0.000118,Europe,Western Europe,Europe,Europe,...,unknown [unknown %],NaN,new AND converted segments,,,,,2029.0,Poland,Poland
6601,Malaysia Singapore Gas Pipeline,,P7105,Malaysia,35,0.500000,Asia,South-eastern Asia,SE Asia,Asia Pacific,...,Petroliam Nasional Bhd [100.00%],NaN,,,,,,1991.0,Malaysia,Singapore


In [141]:
pipes_df_touse.head()

,PipelineNetworkGrouping,PipelineName,SegmentName,Wiki,ProjectID,Status,Status [ref],Researcher,LastUpdated,Fuel,...,PCI6ProjectCode,PipelineDirectionality,QCCOwner,QCCOwner [ref],ImpactedByRussiaUkraineInvasion,EGTImport,ChinesePipelineType,ChineseNetworkPrimary,ChineseNetworkSecondary,ChineseClassificationNotes
69,,Mier Monterrey Gas Pipeline,,https://www.gem.wiki/Mier_Monterrey_Gas_Pipeline,P0215,operating,,GC,2023-08-23,Gas,...,,,,,,,,,,
106,Sur de Texas-Coatzacoalcos Pipeline Network,Sur de Texas-Tuxpan Gas Pipeline,,https://www.gem.wiki/Sur_de_Texas-Tuxpan_Gas_P...,P0257,operating,,GC,2023-08-23,Gas,...,,,,,,,,,,
139,,Nueva Era Pipeline,,https://www.gem.wiki/Nueva_Era_Pipeline,P0290,operating,,GC,2023-08-23,Gas,...,,,,,,,,,,
177,,Energia Mayakan Pipeline,,https://www.gem.wiki/Energia_Mayakan_Pipeline,P0339,operating,,GC,2024-08-20,Gas,...,,,,,,,,,,
191,,Paso Norte Pipeline,,https://www.gem.wiki/Paso_Norte_Pipeline,P0382,shelved,,GC,2023-08-23,Gas,...,,,,,,,,,,


### sum LengthMergedKmByCountry and MergedKmByRegion

In [142]:
status_list = ['proposed', 
               'construction', 
               'shelved', 
               'cancelled', 
               'operating', 
               'idle', 
               'mothballed', 
               'retired']
country_list = sorted(list(set(country_ratios_df_touse['Country'])))
region_list = sorted(list(set(country_ratios_df_touse['Region'])))

In [143]:
excel_status_list = ['proposed', 
                     'construction', 
                     'in development (proposed + construction)', 
                     'shelved', 
                     'cancelled', 
                     'operating', 
                     'idle', 
                     'mothballed', 
                     'retired']
excel_status_list_with_countries = ['Country']+excel_status_list

In [144]:
country_ratios_df_subset_status

,PipelineName,SegmentName,ProjectID,Country,LengthEstimateKmByCountry,LengthPerCountryFraction,Region,SubRegion,RegionOld,PipelineBubbleRegion,...,Parent,H2Status,H2Type,CancelledYear,ProposalYear,ConstructionYear,ShelvedYear,StartYearEarliest,StartCountry,EndCountry


In [145]:
country_ratios_df_subset = country_ratios_df_touse.loc[country_ratios_df_touse['Fuel']==fuel_type]

km_by_country = pandas.DataFrame(columns=status_list, index=country_list)
km_by_region = pandas.DataFrame(columns=status_list, index=multiindex_region_subregion)

print('===country-level calculations===')
for status in status_list:
    print(status)
    country_ratios_df_subset_status = country_ratios_df_subset[country_ratios_df_subset['Status']==status]
    km_by_country[status] = country_ratios_df_subset_status.groupby('Country')['LengthMergedKmByCountry'].sum()

print('===regional calculations===')
for status in status_list:
    print(status)
    country_ratios_df_subset_status = country_ratios_df_subset[country_ratios_df_subset['Status']==status]
    km_by_region[status] = country_ratios_df_subset_status.groupby(['Region','SubRegion'])['LengthMergedKmByCountry'].sum()

# fille NaN with 0.0
km_by_region = km_by_region.fillna(0)
km_by_country = km_by_country.fillna(0)

km_by_region['in development (proposed + construction)'] = km_by_region[['proposed','construction']].sum(axis=1)
km_by_country['in development (proposed + construction)'] = km_by_country[['proposed','construction']].sum(axis=1)

km_by_country = km_by_country[excel_status_list]
km_by_region = km_by_region[excel_status_list]

km_by_region.index.names = ['Region','Subregion']
km_by_country.index.name = 'Country'

km_by_region.loc['Total',:] = km_by_region.sum(axis=0).values
km_by_country.loc['Total',:] = km_by_country.sum(axis=0).values

# drop all-zero rows
km_by_country = km_by_country.loc[~(km_by_country==0).all(axis=1)]

km_by_country.replace(0,'',inplace=True)
km_by_region.replace(0,'',inplace=True)

km_by_region.to_excel(excel_writer, 'Kilometers by region')
km_by_country.to_excel(excel_writer, 'Kilometers by country')

===country-level calculations===
proposed
construction
shelved
cancelled
operating
idle
mothballed
retired
===regional calculations===
proposed
construction
shelved
cancelled
operating
idle
mothballed
retired


/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_12383/2198785467.py:40: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  km_by_region.to_excel(excel_writer, 'Kilometers by region')
/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_12383/2198785467.py:41: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  km_by_country.to_excel(excel_writer, 'Kilometers by country')


In [146]:
km_by_region

,,proposed,construction,in development (proposed + construction),shelved,cancelled,operating,idle,mothballed,retired
Region,Subregion,,,,,,,,,
Americas,Latin America and the Caribbean,15555.28,2885.0,18440.28,4734.05,3387.28,62035.79,331.0,224.0,
Total,,15555.28,2885.0,18440.28,4734.05,3387.28,62035.79,331.0,224.0,


## pipeline km by parent company (owner) and project status

### first check that there are no missing projectids

In [147]:
owner_parent_calculations_df = pandas.DataFrame()
# needs country, km in each country columns as well

for idx,row in country_ratios_df_subset.iterrows():
    #print(row.ProjectID)
    parent_string = row.Parent
    #print(parent_string)

    #print(parent_string)
    # if the first letter of the parent_string is a chinese character, and it ends with [100.00%],
    # that means it's a non-researched QCC owner; keep as-is
    if re.findall(r'[\u4e00-\u9fff]+', parent_string[:1]) != [] and parent_string[-9:]=='[100.00%]':
        parent_list = [parent_string.split(' [100.00%]')[0]]
        percent_list = [1.0]
    # otherwise do the rest of the thing
    else:
        parent_list = re.sub(' \[.*?\]', '', parent_string).split('; ') # all entries must have a Owner [%] syntax
        percent_list = [float(i.rstrip('%'))/100. for i in re.findall('\\d+(?:\\.\\d+)?%', parent_string)]

    if parent_list.__len__()!=percent_list.__len__():
        #print(parent_list)
        if percent_list==[]:
            percent_list = [1/parent_list.__len__() for i in parent_list]
        else:
            nmissing = parent_list.__len__()-percent_list.__len__()
            # distribute nans evenly
            total = numpy.nansum(percent_list)
            leftover = 1-total
            percent_list += [leftover/nmissing]*nmissing
    for p_idx,parent in enumerate(parent_list):
        owner_parent_calculations_df = pandas.concat([owner_parent_calculations_df, 
                                                      pandas.DataFrame([{'Parent':parent, 'ProjectID':row.ProjectID, 
                                                                         'FractionOwnership':percent_list[p_idx],
                                                                         # 'ParentHQCountry':parent_metadata_df.loc[parent,'ParentHQCountry'],
                                                                         # 'ParentHQRegion':parent_metadata_df.loc[parent,'ParentHQRegion'],
                                                                         'Country':row.Country,
                                                                         'Status':row.Status,
                                                                         'LengthMergedKmByCountry':row.LengthMergedKmByCountry}])])

owner_parent_calculations_df['KmOwnership'] = owner_parent_calculations_df.FractionOwnership*owner_parent_calculations_df.LengthMergedKmByCountry

In [148]:
owner_parent_calculations_df

,Parent,ProjectID,FractionOwnership,Country,Status,LengthMergedKmByCountry,KmOwnership
0,Kinder Morgan Inc,P0184,1.0,Mexico,operating,0.14,0.140
0,Kinder Morgan Inc,P0215,1.0,Mexico,operating,144.81,144.810
0,TC Energy Corp,P0257,0.6,Mexico,operating,759.46,455.676
0,Sempra Energy,P0257,0.4,Mexico,operating,759.46,303.784
0,Kinder Morgan Inc,P0258,1.0,Mexico,operating,8.07,8.070
...,...,...,...,...,...,...,...
0,unknown,P6611,1.0,Brazil,proposed,412.34,412.340
0,unknown,P6611,1.0,Argentina,proposed,116.85,116.850
0,unknown,P6611,1.0,Paraguay,proposed,520.81,520.810
0,unknown,P6841,1.0,Peru,operating,408.00,408.000


In [149]:
excel_status_list_with_countries

['Country',
 'proposed',
 'construction',
 'in development (proposed + construction)',
 'shelved',
 'cancelled',
 'operating',
 'idle',
 'mothballed',
 'retired']

In [150]:
owners_km_by_status_df

,proposed,construction,in development (proposed + construction),shelved,cancelled,operating,idle,mothballed,retired,Country
Parent,,,,,,,,,,
APA Corp,,,,,,236.75,,,,
Atlas Petroleum International Ltd,,,,,,14.0,,,,
BASF SE,,,,,,160.0,,,,
BP PLC,,,,,,275.145,,,,
Bharat Petroleum Corp Ltd,4.5,,4.5,,,,,,,
...,...,...,...,...,...,...,...,...,...,...
Transnet SOC Ltd,3513.96,,3513.96,,,600.0,,,,
Tunisian Company of Electricity and Gas,,,,,,294.0,,,,
small shareholder(s),,,,,,133.348,,,,


In [151]:
#unique_owner_list = owner_parent_calculations_df.Parent.sort_values().unique().tolist()

##################################################
# create km count by owner, status
##################################################
owners_km_by_status_df = \
    owner_parent_calculations_df.groupby(
    ['Parent',
     # 'ParentHQCountry',
     'Status'])[['KmOwnership']].sum().unstack().droplevel(axis=1, level=[0])

owners_km_by_status_df = owners_km_by_status_df.reindex(columns=status_list)
owners_km_by_status_df = owners_km_by_status_df.reset_index().set_index('Parent')
owners_km_by_status_df.columns.name = None

owners_km_by_status_df['in development (proposed + construction)'] = owners_km_by_status_df[['proposed','construction']].sum(axis=1)

owners_km_by_status_df = owners_km_by_status_df.rename(columns={'Parent':'Parent Company',
                                                                          # 'ParentHQCountry':'Country'
                                                               })
# rearrange the order of the columns for output
owners_km_by_status_df = owners_km_by_status_df[excel_status_list]#_with_countries]

# totals_row = owners_ntrains_by_status_df.sum(axis=0, min_count=0)
# totals_row.name = 'Total'
# owners_ntrains_by_status_df = owners_ntrains_by_status_df.append(totals_row)
owners_km_by_status_df.loc['Total',:] = owners_km_by_status_df.sum(axis=0, min_count=0).values
owners_km_by_status_df.loc['Total','Country'] = ''

owners_km_by_status_df = owners_km_by_status_df.replace(numpy.nan, '')
owners_km_by_status_df = owners_km_by_status_df.replace(0, '')

owners_km_by_status_df.to_excel(excel_writer, sheet_name='Kilometers by owner')

### pipeline km by start year, type

In [152]:
pipes_started = pipes_df_touse.copy()
#pipes_started['StartYearLatest'].replace(numpy.nan,'',inplace=True)

if fuel_type == 'Gas':
    pipes_started = pipes_started[(pipes_started['Status'].isin(['operating'])) &
                              (pipes_started['Fuel']=='Gas')]
if fuel_type == 'Oil':
    pipes_started = pipes_started[(pipes_started['Status'].isin(['operating'])) &
                              (pipes_started['Fuel']=='Oil')]
if fuel_type == 'NGL':
    pipes_started = pipes_started[(pipes_started['Status'].isin(['operating'])) &
                              (pipes_started['Fuel']=='NGL')]

pipes_started_sum = pipes_started.groupby('StartYearEarliest')['LengthMergedKm'].sum()

In [153]:
pipes_started_sum

StartYearEarliest
1950.0     2470.00
1951.0      772.00
1960.0     1454.00
1965.0     4590.00
1969.0      357.00
1970.0     2046.00
1972.0      529.28
1974.0      225.00
1977.0     1222.50
1981.0     1401.00
1984.0      340.00
1986.0     1366.80
1988.0     1958.00
1994.0      120.00
1995.0       76.00
1997.0     4566.00
1998.0      846.20
1999.0    15443.00
2000.0      437.00
2001.0      311.00
2002.0     1262.80
2003.0      817.84
2004.0     1410.00
2005.0       76.50
2006.0     1253.30
2007.0     1769.00
2008.0      328.04
2009.0      841.60
2010.0      450.05
2011.0      724.00
2012.0       54.80
2013.0      384.46
2014.0     1204.56
2016.0     1840.00
2017.0      263.00
2018.0     1418.87
2019.0     2014.00
2020.0      621.00
2021.0      650.00
2022.0      326.00
2023.0      684.20
2024.0      529.00
Name: LengthMergedKm, dtype: float64

In [154]:
if fuel_type == 'Gas':
    km_by_start_year = pandas.DataFrame(index=list(range(1980,2025)), columns=['Gas pipeline km'])
    km_by_start_year.index.name = 'Start year'
    km_by_start_year['Gas pipeline km'] = pipes_started_sum
    km_by_start_year.replace(numpy.nan,0,inplace=True)

if fuel_type == 'Oil':
    km_by_start_year = pandas.DataFrame(index=list(range(1980,2025)), columns=['Oil pipeline km'])
    km_by_start_year.index.name = 'Start year'
    km_by_start_year['Oil pipeline km'] = pipes_started_sum
    km_by_start_year.replace(numpy.nan,0,inplace=True)

if fuel_type == 'NGL':
    km_by_start_year = pandas.DataFrame(index=list(range(1980,2025)), columns=['NGL pipeline km'])
    km_by_start_year.index.name = 'Start year'
    km_by_start_year['NGL pipeline km'] = pipes_started_sum
    km_by_start_year.replace(numpy.nan,0,inplace=True)

km_by_start_year.loc['Total',:] = km_by_start_year.sum(axis=0)

km_by_start_year.to_excel(excel_writer, 'Kilometers by start year')
#km_by_start_year

/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_12383/652397138.py:21: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  km_by_start_year.to_excel(excel_writer, 'Kilometers by start year')


## save excel file

In [155]:
excel_writer.close()

## calculating stats for landing page

In [156]:
# number of projects tracked in total
print(pipes_df_touse.loc[pipes_df_touse.Fuel==fuel_type].shape[0], 'gas pipeline projects tracked')
print(pipes_df_touse.loc[pipes_df_touse.Fuel==fuel_type]['LengthMergedKm'].sum()/1e6, 'M km tracked')

179 gas pipeline projects tracked
0.0897156 M km tracked


In [157]:
# number of projects tracked in total
print(pipes_df_touse.loc[(pipes_df_touse.Fuel==fuel_type)&
                        (pipes_df_touse.Status.isin(['proposed','construction']))].shape[0], 'gas pipeline projects tracked')
print(pipes_df_touse.loc[(pipes_df_touse.Fuel==fuel_type)&
                        (pipes_df_touse.Status.isin(['proposed','construction']))]['LengthMergedKm'].sum()/1e3, 'K km tracked')

37 gas pipeline projects tracked
18.439 K km tracked
